In [39]:
# 본 자료는 TTS 커널을 이용해 주세요.

In [40]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [41]:
from TTS.api import TTS
from pathlib import Path

import pandas as pd

from IPython.display import display, Audio
import ipywidgets as w

In [42]:
def xtts_to_file(text: str, speaker_wavs, out_path, *, speaker_id=None, **overrides):
    kw = dict(
        speaker_wav=speaker_wavs,
        language="ko",
        temperature=0.1,
        top_p=0.6,
        top_k=50,
        repetition_penalty=5.0,
        length_penalty=1.0,
        split_sentences=False,
    )
    kw.update(overrides)  # 호출 시 일부만 덮어쓰기 가능

    out_path = str(out_path)
    tts.tts_to_file(text=text, file_path=out_path, **kw)
    return out_path

In [43]:
ROOT = Path.home() / "data_team_tts"
CSV_PATH = ROOT / "refs.csv"
AUDIO_ROOT = ROOT / "audio"

OUT_DIR = Path("./tts_out")

# 특정 화자 폴더 지정 (예: student01)
SPEAKER_ID = "student01"

In [44]:
OUT_DIR.mkdir(exist_ok=True, parents=True)

# 1) refs.csv 로드
df = pd.read_csv(CSV_PATH, keep_default_na=False, na_filter=False)
df["file"] = df["file"].astype(str).str.strip()
df["transcript"] = df["transcript"].astype(str).str.strip()

model_name = "tts_models/multilingual/multi-dataset/xtts_v2"
tts = TTS(model_name, gpu=True)
print("tts 로딩 완료")

/home/data/bootcamp/user2/venvs/tts/lib/python3.10/site-packages/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


/home/data/bootcamp/user2/venvs/tts/lib/python3.10/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.speakers = torch.load(speaker_file_path)
/home/data/bootcamp/user2/venvs/tts/lib/python3.10/site-packages/TTS/utils/io.py:54: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location, **kwargs)


tts 로딩 완료


In [45]:
# 2) 특정화자로 필터링
df_spk = df[df["file"].str.startswith(f"{SPEAKER_ID}/")].copy()

# 3) wav 경로 절대경로로 변환 + 정렬
df_spk["wav_path"] = df_spk["file"].apply(lambda p: (AUDIO_ROOT / p).resolve())
df_spk = df_spk[df_spk["wav_path"].apply(lambda p: Path(p).exists())].copy()
df_spk = df_spk.sort_values("file")

speaker_wavs = [str(p) for p in df_spk["wav_path"].tolist()]

print(f"[{SPEAKER_ID}] wav count:", len(speaker_wavs))
print("first 3 wavs:", speaker_wavs[:3])

[student01] wav count: 10
first 3 wavs: ['/home/data/bootcamp/user2/data_team_tts/audio/student01/0001.wav', '/home/data/bootcamp/user2/data_team_tts/audio/student01/0002.wav', '/home/data/bootcamp/user2/data_team_tts/audio/student01/0003.wav']


In [46]:
# 5) 합성 테스트
text = "안녕하세요. 여기에 문장을 입력하면 음성이 생성됩니다. 제 목소리 이상해요 왜"
out_wav = OUT_DIR / f"xtts_{SPEAKER_ID}_multi_ref.wav"

xtts_to_file(text, speaker_wavs, out_wav, temperature=0.1, top_p=0.8)
Audio(str(out_wav))

['안녕하세요. 여기에 문장을 입력하면 음성이 생성됩니다. 제 목소리 이상해요 왜']
 > Processing time: 2.3913559913635254
 > Real-time factor: 0.3359758870015148


In [47]:
text_box = w.Textarea(
    value="안녕하세요. 여기에 문장을 입력하면 음성이 생성됩니다.",
    description="Text",
    layout=w.Layout(width="100%", height="120px")
)
btn = w.Button(description="Generate WAV", button_style="success")
out = w.Output()

def on_click(_):
    with out:
        out.clear_output()
        text = text_box.value.strip()
        if not text:
            print("텍스트를 입력하세요.")
            return
        p = OUT_DIR / "ui_demo.wav"
        xtts_to_file(text, speaker_wavs, p, temperature=0.1, top_p=0.8)
        print("Saved:", p)
        display(Audio(str(p)))

btn.on_click(on_click)

display(text_box, btn, out)

Textarea(value='안녕하세요. 여기에 문장을 입력하면 음성이 생성됩니다.', description='Text', layout=Layout(height='120px', width='100%…

Button(button_style='success', description='Generate WAV', style=ButtonStyle())

Output()